# Baseline training on Colab

Select **Runtime ? Change runtime type ? GPU**, then run the cells in order.

The setup cell creates a Python 3.10 environment automatically using `uv` and installs the single `requirements.txt`. Project commands use that environment; the Colab notebook kernel stays as provided.

`data/`, `weights/` and `results/` are kept on Google Drive, so downloads, embeddings and trained models survive between sessions. Start with `STATE = "DE"` in the setup cell.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Private repositories require a token with access to this repository.
# Paste it into the hidden prompt, or leave blank for a public repository.
import base64
import getpass
import os
import subprocess
from pathlib import Path

REPO = "https://github.com/aaronteork/CS5284_GML.git"
BRANCH = "feature/baseline"
REPO_DIR = Path("/content/CS5284_GML")

def prepare_repository():
    if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout. Choose another directory.")
    token = getpass.getpass("GitHub token (leave blank for a public repo): ").strip()
    git_env = os.environ.copy()
    git_env["GIT_TERMINAL_PROMPT"] = "0"
    if token:
        encoded = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        # Authentication is passed only to this subprocess, never saved in the URL.
        git_env.update(GIT_CONFIG_COUNT="2",
                       GIT_CONFIG_KEY_0="credential.helper", GIT_CONFIG_VALUE_0="",
                       GIT_CONFIG_KEY_1="http.https://github.com/.extraheader",
                       GIT_CONFIG_VALUE_1=f"Authorization: Basic {encoded}")
    if (REPO_DIR / ".git").is_dir():
        branch = subprocess.check_output(["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True).strip()
        if branch != BRANCH:
            raise RuntimeError(f"Existing checkout is on {branch!r}; expected {BRANCH!r}.")
        command = ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH]
    else:
        command = ["git", "clone", "--branch", BRANCH, REPO, str(REPO_DIR)]
    try:
        subprocess.run(command, env=git_env, check=True)
    except subprocess.CalledProcessError:
        raise RuntimeError("GitHub clone/pull failed. Check the repository URL, branch, and token access. "
                           "No later setup cells should be run until this succeeds.") from None
    os.chdir(REPO_DIR)
    print(f"Ready: {REPO_DIR} ({BRANCH})")

prepare_repository()


In [ ]:
# Keep data, weights and results on Drive
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/cs5284_project")

for name in ["data", "weights", "results"]:
    target = DRIVE_DIR / name
    target.mkdir(parents=True, exist_ok=True)
    link = Path(name)
    if not link.is_symlink():
        !rm -rf {link}
        link.symlink_to(target)
    print(f"{name}/ -> {target}")

In [ ]:
import sys
import subprocess
from pathlib import Path

def run_live(command):
    """Forward subprocess output into the notebook as it arrives."""
    import os
    child_env = dict(os.environ, PYTHONUNBUFFERED="1")
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, env=child_env) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

STATE = "DE"  # Change to "MD" to run Maryland after Delaware.
TRAIN_PYTHON = "/content/cs5284-env/bin/python"
WALK_WORKERS = 0  # Automatic CPU detection (up to 8). Use 1 for original serial walks.

run_live([sys.executable, "-m", "pip", "install", "-q", "uv"])
if not Path(TRAIN_PYTHON).exists():
    run_live([sys.executable, "-m", "uv", "venv", "--python", "3.10",
                    "--seed", "/content/cs5284-env"])
run_live([TRAIN_PYTHON, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
run_live([TRAIN_PYTHON, "-c", "import torch; print('GPU:', torch.cuda.is_available()); "
                "assert torch.cuda.is_available(), 'Select a GPU runtime before training'"])


## Node2Vec embeddings (GraphSAGE only)

Uses only the original `karateclub.Node2Vec` backend installed from `requirements.txt`. Generates walks in parallel on CPU using karateclub's original walk function, then runs its original embedding training. A progress bar shows completed walk nodes. `WALK_WORKERS = 0` automatically uses available CPUs (up to 8); use `1` for original serial execution. More workers require more RAM.

Old custom skip-gram files must be regenerated with `--force`. When moving embeddings between machines, copy both the `.npy` and `.json` files. There is no fallback implementation.

If Dataverse returns HTTP 403, download `DE.zip` (or `MD.zip`) from https://doi.org/10.7910/DVN/V71K5R in your browser, upload it to `My Drive/cs5284_project/data/`, then rerun the generation cell. Existing data and outputs do not need to be cleared.

Parallel walks use fixed chunks and per-chunk seeds, so they differ from the original serial seed sequence. Execution settings are recorded in the `.json` file. Compatible older serial embeddings are reused; use `--force` in the generation command only if you want to regenerate them.


In [ ]:
run_live([TRAIN_PYTHON, "-u", "-m", "src.data.node2vec", "--state", STATE,
          "--walk-workers", str(WALK_WORKERS)])


In [ ]:
# Check embeddings using the project environment
embedding_check = """
import sys
# Sanity check: connected intersections should have more similar embeddings than random pairs
import numpy as np
import torch

for state in [sys.argv[1]]:
    emb = np.load(f"data/embeddings/node2vec/{state}_128.npy")
    emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)
    src, dst = torch.load(f"data/{state}/adj_matrix.pt", weights_only=False).coalesce().indices().numpy()

    rng = np.random.default_rng(0)
    edges = rng.choice(len(src), 5000)
    others = rng.integers(0, len(emb), 5000)
    neighbour = (emb[src[edges]] * emb[dst[edges]]).sum(1).mean()
    random = (emb[src[edges]] * emb[others]).sum(1).mean()
    print(f"{state}: neighbour cosine {neighbour:.3f} vs random {random:.3f}")
    assert neighbour > random + 0.2, f"{state} embeddings look untrained"
"""
run_live([TRAIN_PYTHON, "-c", embedding_check, STATE])


## Train

Run a 2-epoch MLP smoke test first, then all four baseline configurations for the selected state (100 epochs, 3 seeds each). Each subprocess stops on failure.

Best validation checkpoints go to `weights/<run>/`; metrics and test predictions go to `results/<run>/`. The smoke test uses a separate run name. To run Maryland later, change `STATE` in setup and rerun from there.

Training displays an epoch progress bar and validation logs. Node2Vec reports cache reuse and shows a walk-generation progress bar in parallel mode, followed by the original gensim embedding training.


In [ ]:
SEEDS = ["0", "1", "2"]
EPOCHS = 100
NODE2VEC = f"data/embeddings/node2vec/{STATE}_128.npy"

run_live([TRAIN_PYTHON, "-u", "-m", "src.model.train", "--state", STATE,
                "--model", "mlp", "--task", "classification", "--epochs", "2",
                "--seeds", "0", "--select-protocol", "all",
                "--run-name", f"{STATE}_mlp_classification_smoke"])

for model in ["mlp", "graphsage"]:
    for task in ["classification", "regression"]:
        command = [TRAIN_PYTHON, "-u", "-m", "src.model.train", "--state", STATE,
                   "--model", model, "--task", task, "--epochs", str(EPOCHS),
                   "--seeds", *SEEDS, "--select-protocol", "all"]
        if model == "graphsage":
            command += ["--node-features", NODE2VEC]
        run_live(command)


## Compare with the paper

In [ ]:
import json
import pandas as pd

# Paper Table 2 (test split, mean over 3 seeds)
PAPER = {
    ("DE", "mlp"): {"AUROC": 0.755, "MAE": 1.4},
    ("DE", "graphsage"): {"AUROC": 0.876, "MAE": 0.3},
    ("MD", "mlp"): {"AUROC": 0.741, "MAE": 0.4},
    ("MD", "graphsage"): {"AUROC": 0.876, "MAE": 0.4},
}

rows = []
for path in sorted(Path("results").glob("*/metrics.json")):
    result = json.load(open(path))
    cfg, test = result["config"], result["summary"]["test"]
    if cfg["run_name"].endswith("_smoke"):
        continue
    metric = "AUROC" if cfg["task"] == "classification" else "MAE"
    row = {"run": cfg["run_name"], "state": cfg["state"], "model": cfg["model"], "metric": metric}
    for protocol in ["all", "balanced"]:
        m = test[protocol][metric]
        row[protocol] = f"{m['mean']:.3f} ± {m['std']:.3f}"
    row["paper"] = PAPER.get((cfg["state"], cfg["model"]), {}).get(metric)
    rows.append(row)

pd.DataFrame(rows)